In [323]:
import pandas as pd
import numpy as  np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [324]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

In [325]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [326]:
df = pd.read_csv('../data/Telco_customer_churn.csv')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [327]:
df.columns

Index(['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
       'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen',
       'Partner', 'Dependents', 'Tenure Months', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value',
       'Churn Score', 'CLTV', 'Churn Reason'],
      dtype='str')

In [328]:
features = [
    "Tenure Months",
    'Total Charges'
]
target = 'Monthly Charges'

X = df[features].copy()
y = df[target]

In [329]:
for column in features:
    X[column] = pd.to_numeric(X[column], errors='coerce')

y = pd.to_numeric(y, errors='coerce')

In [330]:
print("Missing Values of X : \n", X.isnull().sum())
print("\nMissing Values of y : ", y.isnull().sum())

Missing Values of X : 
 Tenure Months     0
Total Charges    11
dtype: int64

Missing Values of y :  0


In [331]:
data = X.copy()
data[target] = y
data = data.dropna()

print("\nDataset Shape After Removing Missing Values:")
print(data.shape)


Dataset Shape After Removing Missing Values:
(7032, 3)


In [332]:
X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [333]:
from sklearn.neighbors import KNeighborsRegressor
model = KNeighborsRegressor(n_neighbors=5)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

In [334]:
# ============================================================
# 16. ACTUAL VS PREDICTED
# ============================================================

comparison = pd.DataFrame({

    "Actual Monthly Charges": y_test.values[:10],

    "Predicted Monthly Charges": y_pred[:10]

})


print("\nActual vs Predicted:")
print(comparison)


Actual vs Predicted:
   Actual Monthly Charges  Predicted Monthly Charges
0                   25.25                      24.77
1                   44.15                      47.79
2                   89.20                      86.71
3                   19.55                      19.74
4                   71.00                      74.18
5                   20.50                      19.87
6                   25.00                      24.32
7                   20.25                      20.04
8                   59.40                      58.94
9                   85.80                      84.71


In [335]:
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2Score = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)

print("Mean Squred Error : ", mse)
print("Mean Absolute Error : ", mae)
print("R2 Score : ", r2Score)
print("Adjusted R2 Score : ", rmse)

Mean Squred Error :  7.899153375977257
Mean Absolute Error :  1.8464250177683017
R2 Score :  0.9912021207205522
Adjusted R2 Score :  2.8105432528209304


In [336]:
k_values = [
    1,
    3,
    5,
    7,
    9,
    11,
    15,
    21
]


In [337]:
for k in k_values:
    model_k = KNeighborsRegressor(n_neighbors=k)
    model_k.fit(X_train_scaled, y_train)
    predictions = model_k.predict(X_test_scaled)

    mScore = mean_squared_error(y_test, predictions)
    mAbsScore = mean_absolute_error(y_test, predictions)
    rScore = r2_score(y_test, predictions)
    rmseScore = np.sqrt(mScore)

    print(

        "K =", k,

        "| RMSE =", rmseScore,

        "| R² =", rScore

    )

K = 1 | RMSE = 3.6723355415565817 | R² = 0.9849795769856766
K = 3 | RMSE = 3.0236554533218145 | R² = 0.9898173209527948
K = 5 | RMSE = 2.8105432528209304 | R² = 0.9912021207205522
K = 7 | RMSE = 2.7690632134654627 | R² = 0.9914598953556015
K = 9 | RMSE = 2.7523510913167826 | R² = 0.9915626684396804
K = 11 | RMSE = 2.7428698086317267 | R² = 0.9916206980564399
K = 15 | RMSE = 2.785123002629741 | R² = 0.991360547676055
K = 21 | RMSE = 2.858647157040389 | R² = 0.9908983829353117


In [338]:
paramGrid = {
    "n_neighbors" : [
        1,
        3,
        5,
        7,
        9,
        11,
        15,
        21
    ]
}

grid_model = GridSearchCV(estimator=KNeighborsRegressor(), param_grid=paramGrid, cv=5, scoring="neg_mean_squared_error")

In [339]:
grid_model.fit(X_train_scaled, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsRegressor()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'n_neighbors': [1, 3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitt

In [340]:
best_model = grid_model.best_estimator_
print("Best Estimator : ", best_model)
best_k = grid_model.best_params_["n_neighbors"]
print("Best K Value : ", best_k)


best_cv_mse = -grid_model.best_score_

best_cv_rmse = np.sqrt(
    best_cv_mse
)

print("\nBest Cross-Validation MSE:")
print(best_cv_mse)

print("\nBest Cross-Validation RMSE:")
print(best_cv_rmse)


Best Estimator :  KNeighborsRegressor(n_neighbors=7)
Best K Value :  7

Best Cross-Validation MSE:
7.689122512471656

Best Cross-Validation RMSE:
2.772926705210878


In [341]:
y_pred_best = best_model.predict(X_test_scaled)

final_mse = mean_squared_error(y_test, y_pred_best)
final_mae = mean_absolute_error(y_test, y_pred_best)
final_rmse = np.sqrt(final_mse)
final_r2 = r2_score(y_test, y_pred_best)

print("Best K Value : ", grid_model.best_score_)
print("Best Mean Squred Error : ", final_mse)
print("Best Mean Absolute Error : ", final_mae)
print("Best R2 Score : ", final_r2)
print("Best Adjusted R2 Score : ", final_rmse)


Best K Value :  -7.689122512471656
Best Mean Squred Error :  7.667711080167676
Best Mean Absolute Error :  1.8330591938267848
Best R2 Score :  0.9914598953556015
Best Adjusted R2 Score :  2.7690632134654627


In [342]:
result = X_test.copy()
result['Actual Monthly Charges'] = y_test
result["Predicted Monthly Charges"] = y_pred_best
result["Absolute Error"] = abs(
    result["Actual Monthly Charges"]
    -
    result["Predicted Monthly Charges"]
)


# ============================================================
# 35. RESET INDEX
# ============================================================

result = result.reset_index()


# ============================================================
# 36. DISPLAY SAMPLE PREDICTIONS
# ============================================================

print("\nSample Predictions:")

print(
    result.head(10)
)


Sample Predictions:
   index  Tenure Months  Total Charges  Actual Monthly Charges  \
0   2478             42        1108.20                   25.25   
1   6783             43        1931.30                   44.15   
2   6125             27        2383.60                   89.20   
3   3052             65        1218.65                   19.55   
4   4097             29        2080.10                   71.00   
5   3223             32         696.80                   20.50   
6   3773             71        1753.00                   25.00   
7   3468             19         383.65                   20.25   
8   3419             33        1952.80                   59.40   
9   1192              3         272.20                   85.80   

   Predicted Monthly Charges  Absolute Error  
0                  24.792857        0.457143  
1                  47.585714        3.435714  
2                  88.714286        0.485714  
3                  19.678571        0.128571  
4                